# 固定済み before / after ペアの Lab 解析

この notebook は **探索を一切しません**。`OUTPUT/selected_pair.json` で固定済みの before / after と ROI を再検証してから Lab 解析だけを行います。

既に探索が終わっているときは、この notebook だけ開いて `OUTPUT` を合わせ、**Run All** してください。


In [1]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_cheek_lab.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')

print('repo root:', Path.cwd())

repo root: C:\Users\mail\work\ikiikimake


## 設定


In [2]:
# 探索済みフォルダ
OUTPUT = Path('outputs/makeup_video_search_wide')
SELECTED = OUTPUT / 'selected_pair.json'
LAB_OUTPUT = OUTPUT / 'lab_selected_pair'

print('selected pair:', SELECTED)
print('lab output   :', LAB_OUTPUT)

selected pair: outputs\makeup_video_search_wide\selected_pair.json
lab output   : outputs\makeup_video_search_wide\lab_selected_pair


## 固定済みペアを検証して解析

元画像と ROI 成果物の SHA-256 が `selected_pair.json` と一致しない場合は停止します。既存の `lab_selected_pair` がある場合も上書きしません。


In [3]:
import hashlib
import json

from analysis.analyze_cheek_lab import analyze_pair

if not SELECTED.is_file():
    raise FileNotFoundError(f'selected_pair.json がありません: {SELECTED}')
if LAB_OUTPUT.exists():
    raise FileExistsError(f'Lab解析出力が既にあります。上書きしません: {LAB_OUTPUT}')

selected = json.loads(SELECTED.read_text(encoding='utf-8'))

def sha256_file(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def checked_file(path: Path, expected_sha256: str, label: str) -> Path:
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise RuntimeError(
            f'{label} の SHA-256 が一致しません。'
            f' expected={expected_sha256} actual={actual} path={path}'
        )
    return path

resolved = {}
for phase in ('before', 'after'):
    item = selected.get(phase)
    if not isinstance(item, dict):
        raise ValueError(f'selected_pair.json に {phase} がありません。')

    image = checked_file(Path(item['image_path']), item['image_sha256'], f'{phase} image')
    roi_dir = Path(item['roi_dir'])
    masks = checked_file(roi_dir / 'roi_masks.npz', item['roi_masks_sha256'], f'{phase} roi_masks')
    checked_file(roi_dir / 'roi_points.json', item['roi_points_sha256'], f'{phase} roi_points')
    checked_file(roi_dir / 'roi_overlay.png', item['roi_overlay_sha256'], f'{phase} roi_overlay')
    resolved[phase] = {'image': image, 'masks': masks}

summary = analyze_pair(
    resolved['before']['image'],
    resolved['after']['image'],
    resolved['before']['masks'],
    resolved['after']['masks'],
    LAB_OUTPUT,
)

print('Lab analysis complete:', LAB_OUTPUT)

Lab analysis complete: outputs\makeup_video_search_wide\lab_selected_pair


## 結果

`delta` は after − before、`delta_minus_forehead` は `(頬 after − before) − (額 after − before)` です。額差し引きは未検証の control 比較で、照明補正やメイク効果の確定値ではありません。


In [4]:
from IPython.display import HTML, display

summary_path = LAB_OUTPUT / 'analysis_summary.json'
if not summary_path.is_file():
    raise FileNotFoundError(summary_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))

rows = [row for row in summary['deltas'] if row['metric'] in ('a_median', 'a_mean')]
if not rows:
    raise RuntimeError('a* の差分結果がありません。')

headers = ('side', 'metric', 'before', 'after', 'delta', 'forehead_delta', 'delta_minus_forehead')
head = ''.join(f'<th>{name}</th>' for name in headers)
body = []
for row in rows:
    cells = []
    for name in headers:
        value = row[name]
        cells.append(f'<td>{value:.3f}</td>' if isinstance(value, float) else f'<td>{value}</td>')
    body.append('<tr>' + ''.join(cells) + '</tr>')

display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'))

side,metric,before,after,delta,forehead_delta,delta_minus_forehead
left_cheek,a_mean,14.141,16.547,2.406,0.067,2.339
left_cheek,a_median,14.000,16.000,2.000,0.000,2.000
right_cheek,a_mean,14.370,16.455,2.085,0.067,2.017
right_cheek,a_median,15.000,17.000,2.000,0.000,2.000


## 生成ファイル


In [5]:
required = [
    'analysis_summary.json',
    'lab_stats.csv',
    'lab_deltas.csv',
    'roi_samples.png',
    'left_cheek_lab_hist.png',
    'right_cheek_lab_hist.png',
    'forehead_lab_hist.png',
]
for name in required:
    path = LAB_OUTPUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path)


outputs\makeup_video_search_wide\lab_selected_pair\analysis_summary.json
outputs\makeup_video_search_wide\lab_selected_pair\lab_stats.csv
outputs\makeup_video_search_wide\lab_selected_pair\lab_deltas.csv
outputs\makeup_video_search_wide\lab_selected_pair\roi_samples.png
outputs\makeup_video_search_wide\lab_selected_pair\left_cheek_lab_hist.png
outputs\makeup_video_search_wide\lab_selected_pair\right_cheek_lab_hist.png
outputs\makeup_video_search_wide\lab_selected_pair\forehead_lab_hist.png
